# StatementIQ

Notebook for exploring and analyzing financial statements.

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np
import requests
import json

In [5]:
PROJECT_DIR = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_DIR = PROJECT_DIR / "data" / "sample_statements"
OUTPUT_DIR = PROJECT_DIR / "outputs"

OLLAMA_URL = "http://localhost:11434"
OLLAMA_MODEL = "gemma4:e4b"

print("Project:", PROJECT_DIR)
print("Model:", OLLAMA_MODEL)

Project: /Users/naitikdonda/Desktop/StatementIQ
Model: gemma4:e4b


In [6]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Sample statements:", DATA_DIR)
print("Outputs:", OUTPUT_DIR)

Sample statements: /Users/naitikdonda/Desktop/StatementIQ/data/sample_statements
Outputs: /Users/naitikdonda/Desktop/StatementIQ/outputs


In [7]:
try:
    response = requests.get(
        f"{OLLAMA_URL}/api/tags",
        timeout=5
    )

    response.raise_for_status()

    print("Ollama connection successful")

except requests.exceptions.ConnectionError:
    print("Ollama is not running. Open the Ollama application.")

except requests.exceptions.Timeout:
    print("Ollama connection timed out.")

except requests.exceptions.RequestException as error:
    print("Connection error:", error)

Ollama connection successful


In [8]:
try:
    response = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={
            "model": OLLAMA_MODEL,
            "messages": [
                {
                    "role": "user",
                    "content": "Reply with exactly: StatementIQ local model ready"
                }
            ],
            "stream": False
        },
        timeout=180
    )

    response.raise_for_status()

    result = response.json()

    print("Model:", result.get("model"))
    print("Response:", result["message"]["content"])

except requests.exceptions.ConnectionError:
    print("Ollama is not running.")

except requests.exceptions.Timeout:
    print("Model timed out. It may be loading or too slow.")

except requests.exceptions.HTTPError as error:
    print("Ollama returned an error:", error)
    print(response.text)

except requests.exceptions.RequestException as error:
    print("Request failed:", error)

Model: gemma4:e4b
Response: StatementIQ local model ready


In [11]:
from ucimlrepo import fetch_ucirepo
from pathlib import Path
import pandas as pd

DATA_DIR = PROJECT_DIR / "data" / "sample_statements"
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [16]:
import requests
from tqdm.auto import tqdm

dataset_url = (
    "https://huggingface.co/datasets/attik/"
    "Online-Retail-II-UCI/resolve/main/online_retail_II.csv"
)

retail_file = DATA_DIR / "online_retail_II.csv"

with requests.get(
    dataset_url,
    stream=True,
    timeout=(15, 120)
) as response:

    response.raise_for_status()

    total_size = int(response.headers.get("content-length", 0))

    with open(retail_file, "wb") as file:
        with tqdm(
            total=total_size,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc="Downloading CSV"
        ) as progress:

            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)
                    progress.update(len(chunk))

print("Downloaded:", retail_file)
print("Size:", round(retail_file.stat().st_size / (1024 ** 2), 2), "MB")

Downloaded: /Users/naitikdonda/Desktop/StatementIQ/data/sample_statements/online_retail_II.csv
Size: 90.46 MB


In [18]:
retail_df = pd.read_csv(
    retail_file,
    low_memory=False
)

print("Dataset loaded successfully")
print("Rows:", len(retail_df))
print("Columns:", retail_df.columns.tolist())

Dataset loaded successfully
Rows: 1067371
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [19]:
display(retail_df.head(10))

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom


In [20]:
retail_df.info(memory_usage="deep")

print(
    "Memory usage:",
    round(
        retail_df.memory_usage(deep=True).sum() / (1024 ** 2),
        2
    ),
    "MB"
)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 385.8 MB
Memory usage: 385.8 MB


In [21]:
print("Missing values by column:")
display(retail_df.isnull().sum())

print(
    "\nExact duplicate rows:",
    retail_df.duplicated().sum()
)

Missing values by column:


Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64


Exact duplicate rows: 34335


In [22]:
print("Date range:")
print(retail_df["InvoiceDate"].min())
print(retail_df["InvoiceDate"].max())

print("\nQuantity summary:")
display(retail_df["Quantity"].describe())

print("\nUnit price summary:")
display(retail_df["Price"].describe())

print("\nTransaction countries:")
display(retail_df["Country"].value_counts().head(10))

Date range:
2009-12-01 07:45:00
2011-12-09 12:50:00

Quantity summary:


count    1.067371e+06
mean     9.938898e+00
std      1.727058e+02
min     -8.099500e+04
25%      1.000000e+00
50%      3.000000e+00
75%      1.000000e+01
max      8.099500e+04
Name: Quantity, dtype: float64


Unit price summary:


count    1.067371e+06
mean     4.649388e+00
std      1.235531e+02
min     -5.359436e+04
25%      1.250000e+00
50%      2.100000e+00
75%      4.150000e+00
max      3.897000e+04
Name: Price, dtype: float64


Transaction countries:


Country
United Kingdom    981330
EIRE               17866
Germany            17624
France             14330
Netherlands         5140
Spain               3811
Switzerland         3189
Belgium             3123
Portugal            2620
Australia           1913
Name: count, dtype: int64

In [23]:
retail_df.columns = retail_df.columns.str.strip()

print("Available columns:")
print(retail_df.columns.tolist())

Available columns:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [24]:
column_map = {}

for col in retail_df.columns:
    key = col.lower().replace("_", " ").strip()

    if key in ["invoice", "invoice no", "invoice number"]:
        column_map[col] = "invoice_id"
    elif key in ["invoicedate", "invoice date", "date"]:
        column_map[col] = "transaction_date"
    elif key in ["price", "unit price", "unitprice"]:
        column_map[col] = "unit_price"
    elif key in ["quantity", "qty"]:
        column_map[col] = "quantity"
    elif key in ["description", "product description"]:
        column_map[col] = "description"
    elif key in ["customer id", "customerid", "customer"]:
        column_map[col] = "customer_id"
    elif key in ["country"]:
        column_map[col] = "country"
    elif key in ["stockcode", "stock code", "product code"]:
        column_map[col] = "product_id"

retail_df = retail_df.rename(columns=column_map)

print("Mapped columns:")
print(retail_df.columns.tolist())

Mapped columns:
['invoice_id', 'product_id', 'description', 'quantity', 'transaction_date', 'unit_price', 'customer_id', 'country']


In [25]:
required_columns = [
    "invoice_id",
    "transaction_date",
    "quantity",
    "unit_price"
]

missing_columns = [
    col for col in required_columns
    if col not in retail_df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("Required transaction columns are available.")

Required transaction columns are available.


In [26]:
retail_df["transaction_date"] = pd.to_datetime(
    retail_df["transaction_date"],
    errors="coerce"
)

retail_df["quantity"] = pd.to_numeric(
    retail_df["quantity"],
    errors="coerce"
)

retail_df["unit_price"] = pd.to_numeric(
    retail_df["unit_price"],
    errors="coerce"
)

print(retail_df[
    ["invoice_id", "transaction_date", "quantity", "unit_price"]
].dtypes)

invoice_id                  object
transaction_date    datetime64[ns]
quantity                     int64
unit_price                 float64
dtype: object


In [27]:
retail_df["transaction_amount"] = (
    retail_df["quantity"] * retail_df["unit_price"]
)

display(
    retail_df[
        [
            "invoice_id",
            "transaction_date",
            "description",
            "quantity",
            "unit_price",
            "transaction_amount"
        ]
    ].head(10)
)

,invoice_id,transaction_date,description,quantity,unit_price,transaction_amount
0,489434,2009-12-01 07:45:00,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,6.95,83.4
1,489434,2009-12-01 07:45:00,PINK CHERRY LIGHTS,12,6.75,81.0
2,489434,2009-12-01 07:45:00,WHITE CHERRY LIGHTS,12,6.75,81.0
3,489434,2009-12-01 07:45:00,"RECORD FRAME 7"" SINGLE SIZE",48,2.10,100.8
4,489434,2009-12-01 07:45:00,STRAWBERRY CERAMIC TRINKET BOX,24,1.25,30.0
5,489434,2009-12-01 07:45:00,PINK DOUGHNUT TRINKET POT,24,1.65,39.6
6,489434,2009-12-01 07:45:00,SAVE THE PLANET MUG,24,1.25,30.0
7,489434,2009-12-01 07:45:00,FANCY FONT HOME SWEET HOME DOORMAT,10,5.95,59.5
8,489435,2009-12-01 07:46:00,CAT BOWL,12,2.55,30.6
9,489435,2009-12-01 07:46:00,"DOG BOWL , CHASING BALL DESIGN",12,3.75,45.0


In [28]:
transactions_clean = retail_df.copy()

print("Original rows:", len(transactions_clean))

Original rows: 1067371


In [29]:
transactions_clean = transactions_clean.dropna(
    subset=[
        "invoice_id",
        "transaction_date",
        "quantity",
        "unit_price"
    ]
)

transactions_clean = transactions_clean[
    (transactions_clean["quantity"] != 0) &
    (transactions_clean["unit_price"] >= 0)
]

transactions_clean = transactions_clean.drop_duplicates()

print("Rows after cleaning:", len(transactions_clean))

Rows after cleaning: 1033031


In [30]:
invoice_text = transactions_clean["invoice_id"].astype(str)

transactions_clean["transaction_type"] = "purchase"

transactions_clean.loc[
    invoice_text.str.upper().str.startswith("C"),
    "transaction_type"
] = "cancellation"

transactions_clean.loc[
    transactions_clean["quantity"] < 0,
    "transaction_type"
] = "return_or_reversal"

print(
    transactions_clean["transaction_type"]
    .value_counts()
)

transaction_type
purchase              1010534
return_or_reversal      22496
cancellation                1
Name: count, dtype: int64


In [31]:
transactions_clean["transaction_amount"] = (
    transactions_clean["quantity"] *
    transactions_clean["unit_price"]
)

transactions_clean["transaction_id"] = (
    transactions_clean["invoice_id"].astype(str)
    + "_"
    + transactions_clean.index.astype(str)
)

if "description" in transactions_clean.columns:
    transactions_clean["description"] = (
        transactions_clean["description"]
        .fillna("Unknown product")
        .astype(str)
        .str.strip()
    )

display(
    transactions_clean[
        [
            "transaction_id",
            "invoice_id",
            "transaction_date",
            "transaction_type",
            "quantity",
            "unit_price",
            "transaction_amount"
        ]
    ].head(10)
)

,transaction_id,invoice_id,transaction_date,transaction_type,quantity,unit_price,transaction_amount
0,489434_0,489434,2009-12-01 07:45:00,purchase,12,6.95,83.4
1,489434_1,489434,2009-12-01 07:45:00,purchase,12,6.75,81.0
2,489434_2,489434,2009-12-01 07:45:00,purchase,12,6.75,81.0
3,489434_3,489434,2009-12-01 07:45:00,purchase,48,2.10,100.8
4,489434_4,489434,2009-12-01 07:45:00,purchase,24,1.25,30.0
5,489434_5,489434,2009-12-01 07:45:00,purchase,24,1.65,39.6
6,489434_6,489434,2009-12-01 07:45:00,purchase,24,1.25,30.0
7,489434_7,489434,2009-12-01 07:45:00,purchase,10,5.95,59.5
8,489435_8,489435,2009-12-01 07:46:00,purchase,12,2.55,30.6
9,489435_9,489435,2009-12-01 07:46:00,purchase,12,3.75,45.0


In [32]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cleaned_file = OUTPUT_DIR / "transactions_clean.csv"

transactions_clean.to_csv(
    cleaned_file,
    index=False
)

print("Cleaned dataset saved to:", cleaned_file)
print("Total records:", len(transactions_clean))
print(
    "Total transaction amount:",
    round(transactions_clean["transaction_amount"].sum(), 2)
)

Cleaned dataset saved to: /Users/naitikdonda/Desktop/StatementIQ/outputs/transactions_clean.csv
Total records: 1033031
Total transaction amount: 19014209.84


In [33]:
import sqlite3

DB_DIR = PROJECT_DIR / "database"
DB_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = DB_DIR / "statementiq.db"

conn = sqlite3.connect(DB_PATH)

print("Database connected:", DB_PATH)

Database connected: /Users/naitikdonda/Desktop/StatementIQ/database/statementiq.db


In [34]:
transactions_clean.to_sql(
    "transactions",
    conn,
    if_exists="replace",
    index=False
)

print("Transactions saved to SQLite.")

Transactions saved to SQLite.


In [35]:
conn.execute("""
CREATE INDEX IF NOT EXISTS idx_transaction_date
ON transactions(transaction_date)
""")

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_invoice_id
ON transactions(invoice_id)
""")

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_transaction_type
ON transactions(transaction_type)
""")

conn.commit()

print("Database indexes created.")

Database indexes created.


In [36]:
db_count = pd.read_sql_query(
    "SELECT COUNT(*) AS total_records FROM transactions",
    conn
)

display(db_count)

,total_records
0,1033031


In [37]:
financial_summary = pd.read_sql_query("""
SELECT
    transaction_type,
    COUNT(*) AS record_count,
    ROUND(SUM(transaction_amount), 2) AS total_amount
FROM transactions
GROUP BY transaction_type
ORDER BY total_amount DESC
""", conn)

display(financial_summary)

,transaction_type,record_count,total_amount
0,purchase,1010534,20476260.45
1,cancellation,1,373.57
2,return_or_reversal,22496,-1462424.18


In [38]:
%pip install pypdf

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [39]:
PDF_DIR = PROJECT_DIR / "data" / "financial_pdfs"
PDF_DIR.mkdir(parents=True, exist_ok=True)

print("Place your financial PDF files here:")
print(PDF_DIR)

Place your financial PDF files here:
/Users/naitikdonda/Desktop/StatementIQ/data/financial_pdfs


In [40]:
from pypdf import PdfReader

print("PDF reader ready.")

PDF reader ready.


In [42]:
pdf_files = sorted(PDF_DIR.glob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError(f"No PDF files found in {PDF_DIR}")

for file in pdf_files:
    print(file.name)

print("\nTotal PDFs:", len(pdf_files))

apple_annual_report_2025.pdf
infosys_annual_report_2025.pdf
microsoft_annual_report_2025.pdf

Total PDFs: 3


In [43]:
from pypdf import PdfReader

all_pdf_pages = []

for pdf_path in pdf_files:
    reader = PdfReader(str(pdf_path))

    for page_number, page in enumerate(reader.pages, start=1):
        page_text = page.extract_text() or ""

        all_pdf_pages.append({
            "document_name": pdf_path.name,
            "page_number": page_number,
            "text": page_text.strip()
        })

print("Total pages processed:", len(all_pdf_pages))

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 37 0 (offset 0)
Ignoring wrong pointing object 39 0 (offset 0)
Ignoring wrong pointing object 41 0 (offset 0)
Ignoring wrong pointing object 43 0 (offset 0)
Ignoring wrong pointing object 45 0 (offset 0)
Ignoring wrong pointing object 47 0 (offset 0)
Ignoring wrong pointing object 66 0 (offset 0)
Ignoring wrong pointing object 109 0 (offset 0)
Ignoring wrong pointing object 129 0 (offset 0)
Ignoring wrong pointing object 191 0 (offset 0)
Ignoring wrong pointing object 193 0 (offset 0)
Ignoring wrong pointing object 195 0 (offset 0)
Ignoring wrong pointing object 206 0 (offset 0)
Ignoring wrong pointing object 279 0 (offset 0)
Ignoring wrong pointing object 287 0 (offset 0)
Ignoring wrong pointing object 292 0 (offset 0)
Ignor

Total pages processed: 529


In [44]:
pdf_pages_df = pd.DataFrame(all_pdf_pages)

display(pdf_pages_df.head(10))

print("Documents:", pdf_pages_df["document_name"].nunique())
print("Total pages:", len(pdf_pages_df))

,document_name,page_number,text
0,apple_annual_report_2025.pdf,1,UNITED STATES\nSECURITIES AND EXCHANGE COMMISS...
1,apple_annual_report_2025.pdf,2,Indicate by check mark whether the Registrant ...
2,apple_annual_report_2025.pdf,3,Apple Inc.\nForm 10-K\nFor the Fiscal Year End...
3,apple_annual_report_2025.pdf,4,This Annual Report on Form 10-K (“Form 10-K”) ...
4,apple_annual_report_2025.pdf,5,Services\nAdvertising\nThe Company’s advertisi...
5,apple_annual_report_2025.pdf,6,The Company’s ability to compete successfully ...
6,apple_annual_report_2025.pdf,7,In addition to Company-owned intellectual prop...
7,apple_annual_report_2025.pdf,8,Item 1A. Risk Factors\nThe following summar...
8,apple_annual_report_2025.pdf,9,The Company’s business can be impacted by poli...
9,apple_annual_report_2025.pdf,10,"Following any interruption to its business, th..."


Documents: 3
Total pages: 529


In [45]:
pdf_pages_df["text_length"] = (
    pdf_pages_df["text"].str.len()
)

extraction_report = (
    pdf_pages_df
    .groupby("document_name")
    .agg(
        total_pages=("page_number", "count"),
        pages_with_text=("text_length", lambda x: (x > 50).sum()),
        total_characters=("text_length", "sum")
    )
    .reset_index()
)

display(extraction_report)

,document_name,total_pages,pages_with_text,total_characters
0,apple_annual_report_2025.pdf,80,80,273154
1,infosys_annual_report_2025.pdf,369,368,1207430
2,microsoft_annual_report_2025.pdf,80,78,249739


In [46]:
EXTRACTED_DIR = OUTPUT_DIR / "extracted_pdf_text"
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)

extracted_file = EXTRACTED_DIR / "pdf_pages.csv"

pdf_pages_df.to_csv(
    extracted_file,
    index=False
)

print("Extracted PDF text saved to:")
print(extracted_file)

Extracted PDF text saved to:
/Users/naitikdonda/Desktop/StatementIQ/outputs/extracted_pdf_text/pdf_pages.csv


In [47]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

print("Chunk size:", CHUNK_SIZE)
print("Chunk overlap:", CHUNK_OVERLAP)

Chunk size: 1000
Chunk overlap: 200


In [48]:
def create_chunks(text, chunk_size=1000, overlap=200):
    text = str(text).strip()

    if not text:
        return []

    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(text):
            break

        start = end - overlap

    return chunks

In [49]:
chunk_records = []

for _, row in pdf_pages_df.iterrows():
    page_chunks = create_chunks(
        row["text"],
        CHUNK_SIZE,
        CHUNK_OVERLAP
    )

    for chunk_index, chunk_text in enumerate(page_chunks):
        chunk_records.append({
            "document_name": row["document_name"],
            "page_number": int(row["page_number"]),
            "chunk_index": chunk_index,
            "chunk_text": chunk_text
        })

chunks_df = pd.DataFrame(chunk_records)

print("Total chunks:", len(chunks_df))

Total chunks: 2296


In [50]:
if chunks_df.empty:
    raise ValueError("No chunks generated. Check PDF text extraction.")

display(
    chunks_df[
        [
            "document_name",
            "page_number",
            "chunk_index",
            "chunk_text"
        ]
    ].head(5)
)

,document_name,page_number,chunk_index,chunk_text
0,apple_annual_report_2025.pdf,1,0,UNITED STATES\nSECURITIES AND EXCHANGE COMMISS...
1,apple_annual_report_2025.pdf,1,1,to Section 12(b) of the Act:\nTitle of each cl...
2,apple_annual_report_2025.pdf,1,2,equired to file reports pursuant to Section 13...
3,apple_annual_report_2025.pdf,2,0,Indicate by check mark whether the Registrant ...
4,apple_annual_report_2025.pdf,2,1,estation to its management’s assessment of the...


In [51]:
chunks_file = EXTRACTED_DIR / "pdf_chunks.csv"

chunks_df.to_csv(
    chunks_file,
    index=False
)

print("Chunks saved to:", chunks_file)
print("Unique documents:", chunks_df["document_name"].nunique())
print("Unique pages:", chunks_df["page_number"].nunique())

Chunks saved to: /Users/naitikdonda/Desktop/StatementIQ/outputs/extracted_pdf_text/pdf_chunks.csv
Unique documents: 3
Unique pages: 369


In [52]:
import sqlite3

conn = sqlite3.connect(DB_PATH)

conn.execute("""
CREATE TABLE IF NOT EXISTS pdf_chunks (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    document_name TEXT,
    page_number INTEGER,
    chunk_index INTEGER,
    chunk_text TEXT
)
""")

conn.commit()

print("PDF chunks table created.")

PDF chunks table created.


In [53]:
chunks_df.to_sql(
    "pdf_chunks",
    conn,
    if_exists="replace",
    index=False
)

print("PDF chunks saved to SQLite.")

PDF chunks saved to SQLite.


In [54]:
chunk_count = pd.read_sql_query(
    "SELECT COUNT(*) AS total_chunks FROM pdf_chunks",
    conn
)

display(chunk_count)

,total_chunks
0,2296


In [55]:
sample_chunks = pd.read_sql_query("""
SELECT document_name, page_number, chunk_index, chunk_text
FROM pdf_chunks
LIMIT 5
""", conn)

display(sample_chunks)

,document_name,page_number,chunk_index,chunk_text
0,apple_annual_report_2025.pdf,1,0,UNITED STATES\nSECURITIES AND EXCHANGE COMMISS...
1,apple_annual_report_2025.pdf,1,1,to Section 12(b) of the Act:\nTitle of each cl...
2,apple_annual_report_2025.pdf,1,2,equired to file reports pursuant to Section 13...
3,apple_annual_report_2025.pdf,2,0,Indicate by check mark whether the Registrant ...
4,apple_annual_report_2025.pdf,2,1,estation to its management’s assessment of the...


In [56]:
document_chunks = pd.read_sql_query("""
SELECT 
    document_name,
    COUNT(*) AS total_chunks,
    COUNT(DISTINCT page_number) AS total_pages
FROM pdf_chunks
GROUP BY document_name
ORDER BY document_name
""", conn)

display(document_chunks)

,document_name,total_chunks,total_pages
0,apple_annual_report_2025.pdf,363,80
1,infosys_annual_report_2025.pdf,1602,369
2,microsoft_annual_report_2025.pdf,331,78


In [59]:
import requests
import json
import subprocess
import time

EMBED_MODEL = "nomic-embed-text"
OLLAMA_URL = "http://localhost:11434"

In [60]:
response = requests.get(
    f"{OLLAMA_URL}/api/tags",
    timeout=10
)

response.raise_for_status()

installed_models = [
    model["name"]
    for model in response.json()["models"]
]

print("Ollama is running.")
print("Installed models:", installed_models)

Ollama is running.
Installed models: ['qwen3:8b', 'qwen2.5:3b', 'gemma4:e4b', 'kimi-k2.7-code:cloud']


In [61]:
if not any(
    model.startswith(EMBED_MODEL)
    for model in installed_models
):
    subprocess.run(
        ["ollama", "pull", EMBED_MODEL],
        check=True
    )

print("Embedding model is ready.")

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling 970aa74c0a90:   0% ▕                  ▏  30 KB/274 MB                  pulling manifest 
pulling 970aa74c0a90:   0% ▕                  ▏  79 KB/274 MB                  pulling manifest 
pulling 970aa74c0a90:   0% ▕                  ▏

Embedding model is ready.


pulling manifest 
pulling 970aa74c0a90: 100% ▕█████████████████ ▏ 274 MB/274 MB  2.3 MB/s      0s
verifying sha256 digest ⠙ pulling manifest 
pulling 970aa74c0a90: 100% ▕█████████████████ ▏ 274 MB/274 MB  2.3 MB/s      0s
verifying sha256 digest 
writing manifest 
success 


In [62]:
test_text = chunks_df.iloc[0]["chunk_text"]

response = requests.post(
    f"{OLLAMA_URL}/api/embed",
    json={
        "model": EMBED_MODEL,
        "input": test_text
    },
    timeout=120
)

response.raise_for_status()

test_embedding = response.json()["embeddings"][0]

print("Embedding dimensions:", len(test_embedding))
print("First 10 values:", test_embedding[:10])

Embedding dimensions: 768
First 10 values: [-0.0014691957, 0.061373826, -0.19795844, -0.03771814, 0.02943749, -0.020016905, 0.055785146, 0.039542258, 0.046412278, 0.01218866]


In [63]:
BATCH_SIZE = 32

chunk_texts = chunks_df["chunk_text"].tolist()
all_embeddings = []

for start in range(0, len(chunk_texts), BATCH_SIZE):

    batch = chunk_texts[start:start + BATCH_SIZE]

    response = requests.post(
        f"{OLLAMA_URL}/api/embed",
        json={
            "model": EMBED_MODEL,
            "input": batch
        },
        timeout=300
    )

    response.raise_for_status()

    batch_embeddings = response.json()["embeddings"]

    all_embeddings.extend(batch_embeddings)

    print(
        f"Processed {len(all_embeddings)} "
        f"of {len(chunk_texts)} chunks"
    )

chunks_df["embedding"] = all_embeddings

chunks_df.to_csv(
    OUTPUT_DIR / "pdf_chunks_embeddings.csv",
    index=False
)

print("All PDF embeddings generated and saved.")

Processed 32 of 2296 chunks
Processed 64 of 2296 chunks
Processed 96 of 2296 chunks
Processed 128 of 2296 chunks
Processed 160 of 2296 chunks
Processed 192 of 2296 chunks
Processed 224 of 2296 chunks
Processed 256 of 2296 chunks
Processed 288 of 2296 chunks
Processed 320 of 2296 chunks
Processed 352 of 2296 chunks
Processed 384 of 2296 chunks
Processed 416 of 2296 chunks
Processed 448 of 2296 chunks
Processed 480 of 2296 chunks
Processed 512 of 2296 chunks
Processed 544 of 2296 chunks
Processed 576 of 2296 chunks
Processed 608 of 2296 chunks
Processed 640 of 2296 chunks
Processed 672 of 2296 chunks
Processed 704 of 2296 chunks
Processed 736 of 2296 chunks
Processed 768 of 2296 chunks
Processed 800 of 2296 chunks
Processed 832 of 2296 chunks
Processed 864 of 2296 chunks
Processed 896 of 2296 chunks
Processed 928 of 2296 chunks
Processed 960 of 2296 chunks
Processed 992 of 2296 chunks
Processed 1024 of 2296 chunks
Processed 1056 of 2296 chunks
Processed 1088 of 2296 chunks
Processed 1120

In [64]:
import numpy as np

In [65]:
embedding_matrix = np.array(
    chunks_df["embedding"].tolist(),
    dtype=np.float32
)

embedding_norms = np.linalg.norm(
    embedding_matrix,
    axis=1,
    keepdims=True
)

embedding_norms[embedding_norms == 0] = 1e-10

normalized_embeddings = (
    embedding_matrix / embedding_norms
)

print("Embedding matrix shape:", embedding_matrix.shape)
print("Total searchable chunks:", len(chunks_df))

Embedding matrix shape: (2296, 768)
Total searchable chunks: 2296


In [66]:
def get_query_embedding(question):

    response = requests.post(
        f"{OLLAMA_URL}/api/embed",
        json={
            "model": EMBED_MODEL,
            "input": question
        },
        timeout=120
    )

    response.raise_for_status()

    query_embedding = np.array(
        response.json()["embeddings"][0],
        dtype=np.float32
    )

    return query_embedding

In [67]:
def search_financial_documents(question, top_k=5):

    query_embedding = get_query_embedding(question)

    query_norm = np.linalg.norm(query_embedding)

    if query_norm == 0:
        raise ValueError("Query embedding is empty.")

    query_embedding = query_embedding / query_norm

    similarities = normalized_embeddings @ query_embedding

    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = chunks_df.iloc[top_indices].copy()

    results["similarity_score"] = similarities[top_indices]

    return results[
        [
            "document_name",
            "page_number",
            "chunk_index",
            "chunk_text",
            "similarity_score"
        ]
    ].reset_index(drop=True)

In [69]:
question = "What are the major sources of revenue?"

search_results = search_financial_documents(
    question,
    top_k=5
)

display(search_results)

/var/folders/zc/0lyqtlrn6nv1wxrv92j_qpz80000gn/T/ipykernel_13784/3353327015.py:12: RuntimeWarning: divide by zero encountered in matmul
  similarities = normalized_embeddings @ query_embedding
/var/folders/zc/0lyqtlrn6nv1wxrv92j_qpz80000gn/T/ipykernel_13784/3353327015.py:12: RuntimeWarning: overflow encountered in matmul
  similarities = normalized_embeddings @ query_embedding
/var/folders/zc/0lyqtlrn6nv1wxrv92j_qpz80000gn/T/ipykernel_13784/3353327015.py:12: RuntimeWarning: invalid value encountered in matmul
  similarities = normalized_embeddings @ query_embedding


,document_name,page_number,chunk_index,chunk_text,similarity_score
0,microsoft_annual_report_2025.pdf,71,2,for more than 10% of revenue for \nfiscal year...,0.689481
1,microsoft_annual_report_2025.pdf,70,1,"buted to our segments. However, due to the int...",0.688429
2,microsoft_annual_report_2025.pdf,42,0,41 \nService and other revenue includes sales ...,0.684737
3,infosys_annual_report_2025.pdf,344,2,"hina, Infosys Public Services and other \nente...",0.674901
4,infosys_annual_report_2025.pdf,321,7,"e services 1,55,395 1,45,285 \nRevenue from ...",0.665709


In [70]:
LLM_MODEL = "gemma4:e4b"

print("Answering model:", LLM_MODEL)

Answering model: gemma4:e4b


In [71]:
response = requests.get(
    f"{OLLAMA_URL}/api/tags",
    timeout=10
)

response.raise_for_status()

installed_models = [
    model["name"]
    for model in response.json()["models"]
]

if not any(
    model.startswith(LLM_MODEL)
    for model in installed_models
):
    raise ValueError(
        f"{LLM_MODEL} is not installed. "
        "Check your Ollama model name."
    )

print("Local answering model is ready.")

Local answering model is ready.


In [72]:
def build_context(search_results):

    context_parts = []

    for _, row in search_results.iterrows():

        source = (
            f"Document: {row['document_name']}\n"
            f"Page: {row['page_number']}\n"
            f"Passage: {row['chunk_text']}"
        )

        context_parts.append(source)

    return "\n\n".join(context_parts)

In [73]:
def ask_statementiq(question, top_k=5):

    retrieved = search_financial_documents(
        question,
        top_k=top_k
    )

    context = build_context(retrieved)

    prompt = f"""
You are StatementIQ, a financial document analysis assistant.

Answer the user's question using only the supplied evidence.

Rules:
- Do not invent financial figures or facts.
- If the evidence is insufficient, say so clearly.
- Cite supporting evidence using [Document Name, Page X].
- Distinguish facts from calculations or interpretations.
- Do not claim that a figure is audited unless the evidence says so.

Evidence:
{context}

Question:
{question}

Answer:
"""

    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            "model": LLM_MODEL,
            "prompt": prompt,
            "stream": False
        },
        timeout=600
    )

    response.raise_for_status()

    answer = response.json()["response"]

    return answer, retrieved

In [74]:
question = "What are the major sources of revenue?"

answer, retrieved_evidence = ask_statementiq(
    question,
    top_k=5
)

print("STATEMENTIQ ANSWER")
print(answer)

print("\nRETRIEVED EVIDENCE")
display(
    retrieved_evidence[
        [
            "document_name",
            "page_number",
            "similarity_score"
        ]
    ]
)

/var/folders/zc/0lyqtlrn6nv1wxrv92j_qpz80000gn/T/ipykernel_13784/3353327015.py:12: RuntimeWarning: divide by zero encountered in matmul
  similarities = normalized_embeddings @ query_embedding
/var/folders/zc/0lyqtlrn6nv1wxrv92j_qpz80000gn/T/ipykernel_13784/3353327015.py:12: RuntimeWarning: overflow encountered in matmul
  similarities = normalized_embeddings @ query_embedding
/var/folders/zc/0lyqtlrn6nv1wxrv92j_qpz80000gn/T/ipykernel_13784/3353327015.py:12: RuntimeWarning: invalid value encountered in matmul
  similarities = normalized_embeddings @ query_embedding


STATEMENTIQ ANSWER
Based on the evidence provided, the major sources of revenue vary between the documents:

**For Microsoft:**
Revenue sources include:
*   Sales from cloud-based solutions, which provide customers with software, services, platforms, and content such as Office 365, Azure, Dynamics 365, and gaming [microsoft_annual_report_2025.pdf, Page 42].
*   Solution support and consulting services [microsoft_annual_report_2025.pdf, Page 42].
*   Sales from online advertising and LinkedIn [microsoft_annual_report_2025.pdf, Page 42].
*   Licenses for on-premises software [microsoft_annual_report_2025.pdf, Page 42].

**For Infosys:**
Revenue sources include:
*   Revenue from products and platforms, such as Finacle – core banking solution, Edge Suite of products, Panaya platform, Stater digital platform, and Infosys McCamish – insurance platform [infosys_annual_report_2025.pdf, Page 321].
*   Revenue generated by Infosys Public Services [infosys_annual_report_2025.pdf, Page 344].
*   R

,document_name,page_number,similarity_score
0,microsoft_annual_report_2025.pdf,71,0.689481
1,microsoft_annual_report_2025.pdf,70,0.688429
2,microsoft_annual_report_2025.pdf,42,0.684737
3,infosys_annual_report_2025.pdf,344,0.674901
4,infosys_annual_report_2025.pdf,321,0.665709


In [75]:
def build_evidence(search_results):

    evidence = []

    for i, (_, row) in enumerate(search_results.iterrows(), start=1):

        evidence.append({
            "source_id": f"E{i}",
            "document": row["document_name"],
            "page": int(row["page_number"]),
            "text": row["chunk_text"]
        })

    return evidence

In [76]:
def format_evidence(evidence):

    context_parts = []

    for item in evidence:

        context_parts.append(
            f"""
Source ID: {item['source_id']}
Document: {item['document']}
Page: {item['page']}
Passage:
{item['text']}
"""
        )

    return "\n".join(context_parts)

In [77]:
def ask_statementiq_v2(question, top_k=8):

    retrieved = search_financial_documents(
        question,
        top_k=top_k
    )

    evidence = build_evidence(retrieved)
    context = format_evidence(evidence)

    prompt = f"""
You are StatementIQ, a financial document analysis assistant.

Answer using only the provided evidence.

IMPORTANT:
1. Separate revenue streams, products, and geographic revenue.
2. Do not describe a product as a revenue stream unless the evidence supports it.
3. Answer only for companies relevant to the question.
4. Do not invent numbers, facts, or citations.
5. Cite each factual claim using [E1], [E2], etc.
6. If evidence is insufficient, explicitly state that.

Evidence:
{context}

Question:
{question}

Answer:
"""

    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            "model": LLM_MODEL,
            "prompt": prompt,
            "stream": False
        },
        timeout=600
    )

    response.raise_for_status()

    answer = response.json()["response"]

    return answer, retrieved, evidence

In [78]:
import re

def validate_citations(answer, evidence):

    cited_ids = re.findall(
        r"\[(E\d+)\]",
        answer
    )

    valid_ids = {
        item["source_id"]
        for item in evidence
    }

    invalid_ids = [
        source_id
        for source_id in cited_ids
        if source_id not in valid_ids
    ]

    citation_report = {
        "citations_found": cited_ids,
        "valid_citations": len(cited_ids) - len(invalid_ids),
        "invalid_citations": invalid_ids,
        "citation_check_passed": len(invalid_ids) == 0
    }

    return citation_report

In [79]:
question = "What are Microsoft's major revenue streams?"

answer, retrieved_evidence, evidence = ask_statementiq_v2(
    question,
    top_k=8
)

citation_report = validate_citations(
    answer,
    evidence
)

print("STATEMENTIQ ANSWER")
print(answer)

print("\nCITATION VALIDATION")
display(pd.DataFrame([citation_report]))

print("\nRETRIEVED SOURCES")
display(
    retrieved_evidence[
        [
            "document_name",
            "page_number",
            "similarity_score"
        ]
    ]
)

/var/folders/zc/0lyqtlrn6nv1wxrv92j_qpz80000gn/T/ipykernel_13784/3353327015.py:12: RuntimeWarning: divide by zero encountered in matmul
  similarities = normalized_embeddings @ query_embedding
/var/folders/zc/0lyqtlrn6nv1wxrv92j_qpz80000gn/T/ipykernel_13784/3353327015.py:12: RuntimeWarning: overflow encountered in matmul
  similarities = normalized_embeddings @ query_embedding
/var/folders/zc/0lyqtlrn6nv1wxrv92j_qpz80000gn/T/ipykernel_13784/3353327015.py:12: RuntimeWarning: invalid value encountered in matmul
  similarities = normalized_embeddings @ query_embedding


STATEMENTIQ ANSWER
Based on the evidence, Microsoft's major revenue streams include:

**Cloud and Software Services:**
*   **Server products and cloud services:** This segment includes revenue from Azure and other cloud services, such as cloud and AI consumption-based services, GitHub cloud services, Nuance Healthcare cloud services, and virtual desktop offerings [E1, E2, E4].
*   **Microsoft 365:** This includes revenue from Microsoft 365 Commercial products and cloud services [E2, E5, E8] and Microsoft 365 Consumer products and cloud services [E2, E8].
*   **Dynamics products and cloud services** [E2, E8].
*   **Enterprise and partner services** [E2].
*   Service and other revenue, which includes sales from cloud-based solutions (such as Office 365, Azure, Dynamics 365, and gaming), solution support, and consulting services [E7].

**Devices and Hardware:**
*   **Windows OEM and Devices revenue:** Revenue comes from sales of Windows Pro and non-Pro licenses sold through the OEM channe

,citations_found,valid_citations,invalid_citations,citation_check_passed
0,"[E2, E7, E6, E2]",4,[],True



RETRIEVED SOURCES


,document_name,page_number,similarity_score
0,microsoft_annual_report_2025.pdf,24,0.742819
1,microsoft_annual_report_2025.pdf,72,0.734748
2,microsoft_annual_report_2025.pdf,21,0.732983
3,microsoft_annual_report_2025.pdf,21,0.722755
4,microsoft_annual_report_2025.pdf,25,0.722663
5,microsoft_annual_report_2025.pdf,15,0.712259
6,microsoft_annual_report_2025.pdf,42,0.711252
7,microsoft_annual_report_2025.pdf,21,0.710950


In [80]:
print("Embedding matrix:", embedding_matrix.shape)
print("NaN values:", np.isnan(embedding_matrix).sum())
print("Infinite values:", np.isinf(embedding_matrix).sum())

invalid_rows = ~np.isfinite(embedding_matrix).all(axis=1)

print("Chunks with invalid embeddings:", invalid_rows.sum())

Embedding matrix: (2296, 768)
NaN values: 0
Infinite values: 0
Chunks with invalid embeddings: 0


In [81]:
valid_mask = np.isfinite(embedding_matrix).all(axis=1)

chunks_df = chunks_df.loc[valid_mask].reset_index(drop=True)
embedding_matrix = embedding_matrix[valid_mask]

print("Valid chunks:", len(chunks_df))
print("Embedding matrix:", embedding_matrix.shape)

Valid chunks: 2296
Embedding matrix: (2296, 768)


In [82]:
embedding_norms = np.linalg.norm(
    embedding_matrix,
    axis=1,
    keepdims=True
)

valid_norms = (
    np.isfinite(embedding_norms)
    & (embedding_norms > 0)
).flatten()

chunks_df = chunks_df.loc[valid_norms].reset_index(drop=True)
embedding_matrix = embedding_matrix[valid_norms]

embedding_norms = embedding_norms[valid_norms]

normalized_embeddings = (
    embedding_matrix / embedding_norms
)

print("Normalized matrix:", normalized_embeddings.shape)
print("All finite:", np.isfinite(normalized_embeddings).all())

Normalized matrix: (2296, 768)
All finite: True


In [83]:
def search_financial_documents(question, top_k=5):

    query_embedding = get_query_embedding(question)

    if not np.isfinite(query_embedding).all():
        raise ValueError("Query embedding contains invalid values.")

    query_norm = np.linalg.norm(query_embedding)

    if not np.isfinite(query_norm) or query_norm == 0:
        raise ValueError("Query embedding has an invalid norm.")

    query_embedding = query_embedding / query_norm

    similarities = normalized_embeddings @ query_embedding

    if not np.isfinite(similarities).all():
        raise ValueError("Similarity scores contain invalid values.")

    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = chunks_df.iloc[top_indices].copy()
    results["similarity_score"] = similarities[top_indices]

    return results[
        [
            "document_name",
            "page_number",
            "chunk_index",
            "chunk_text",
            "similarity_score"
        ]
    ].reset_index(drop=True)

In [84]:
search_results = search_financial_documents(
    "What are Microsoft's major revenue streams?",
    top_k=8
)

display(search_results)

print(
    "All scores finite:",
    np.isfinite(search_results["similarity_score"]).all()
)

/var/folders/zc/0lyqtlrn6nv1wxrv92j_qpz80000gn/T/ipykernel_13784/2813527097.py:15: RuntimeWarning: divide by zero encountered in matmul
  similarities = normalized_embeddings @ query_embedding
/var/folders/zc/0lyqtlrn6nv1wxrv92j_qpz80000gn/T/ipykernel_13784/2813527097.py:15: RuntimeWarning: overflow encountered in matmul
  similarities = normalized_embeddings @ query_embedding
/var/folders/zc/0lyqtlrn6nv1wxrv92j_qpz80000gn/T/ipykernel_13784/2813527097.py:15: RuntimeWarning: invalid value encountered in matmul
  similarities = normalized_embeddings @ query_embedding


,document_name,page_number,chunk_index,chunk_text,similarity_score
0,microsoft_annual_report_2025.pdf,24,1,growth \n \nRevenue from Azure and other clo...,0.742819
1,microsoft_annual_report_2025.pdf,72,0,"71 \nRevenue, classified by significant produc...",0.734748
2,microsoft_annual_report_2025.pdf,21,1,and Results of Operations” in our Annual Repor...,0.732983
3,microsoft_annual_report_2025.pdf,21,3,mics 365 revenue growth of \n19%. \n• Server ...,0.722755
4,microsoft_annual_report_2025.pdf,25,2,"ating Income $ 14,166 $ 11,959 ...",0.722663
5,microsoft_annual_report_2025.pdf,15,1,"d in-game content), Xbox Game Pass and other s...",0.712259
6,microsoft_annual_report_2025.pdf,42,0,41 \nService and other revenue includes sales ...,0.711252
7,microsoft_annual_report_2025.pdf,21,2,devices. Our most significant expenses are rel...,0.710950


All scores finite: True


In [85]:
question = "What revenue streams does Microsoft report?"

answer, retrieved_evidence, evidence = ask_statementiq_v2(
    question,
    top_k=8
)

print(answer)

/var/folders/zc/0lyqtlrn6nv1wxrv92j_qpz80000gn/T/ipykernel_13784/2813527097.py:15: RuntimeWarning: divide by zero encountered in matmul
  similarities = normalized_embeddings @ query_embedding
/var/folders/zc/0lyqtlrn6nv1wxrv92j_qpz80000gn/T/ipykernel_13784/2813527097.py:15: RuntimeWarning: overflow encountered in matmul
  similarities = normalized_embeddings @ query_embedding
/var/folders/zc/0lyqtlrn6nv1wxrv92j_qpz80000gn/T/ipykernel_13784/2813527097.py:15: RuntimeWarning: invalid value encountered in matmul
  similarities = normalized_embeddings @ query_embedding


Microsoft reports revenue from several streams, including:

**General Revenue Streams**
*   Offering a range of cloud-based solutions, content, and other services to people and businesses [E1].
*   Licensing and supporting an array of software products [E1].
*   Delivering relevant online advertising to a global audience [E1].
*   Designing and selling devices [E1].

**Revenue Classified by Product and Service Offerings**
Revenue is classified by categories such as [E2]:
*   Server products and cloud services [E2].
*   Microsoft 365 Commercial products and cloud services [E2], which includes subscriptions for Microsoft 365 Commercial, Enterprise Mobility + Security, and various cloud portions like Power BI, Exchange, SharePoint, Microsoft Teams, and Microsoft 365 Copilot [E6].
*   Microsoft 365 Consumer products and cloud services [E2].
*   Gaming [E2], which includes revenue from Xbox content and services like Xbox Game Pass, games, and Xbox Cloud Gaming [E5].
*   LinkedIn [E2].
*   W

In [86]:
for item in evidence:

    print("=" * 80)
    print("SOURCE:", item["source_id"])
    print("DOCUMENT:", item["document"])
    print("PAGE:", item["page"])
    print(item["text"])

SOURCE: E1
DOCUMENT: microsoft_annual_report_2025.pdf
PAGE: 21
and Results of Operations” in our Annual Report on Form 10-K for the year ended June 30, 2024 and our Form 8-K filed on 
December 3, 2024.  
OVERVIEW  
Microsoft is a technology company committed to making digital technology and artificial intelligence (“AI”) available broadly 
and doing so responsibly, with a mission to empower every person and every organization on the planet to achieve more. 
We create platforms and tools, powered by AI, that deliver innovative solutions that meet the evolving needs of our 
customers.  
We generate revenue by offering a wide range of cloud -based solutions, content, and other services to people and 
businesses; licensing and supporting an array of software products; delivering relevant online advertising to a global 
audience; and designing and selling devices. Our most significant expenses are related to compensating employees; 
supporting and investing in our cloud-based services, incl

In [87]:
cited_ids = set(
    re.findall(r"\[(E\d+)\]", answer)
)

retrieved_ids = {
    item["source_id"]
    for item in evidence
}

missing_sources = cited_ids - retrieved_ids

print("Cited source IDs:", cited_ids)
print("Missing source IDs:", missing_sources)

if missing_sources:
    print("WARNING: Answer cites sources not provided to the model.")
else:
    print("All citation IDs refer to retrieved evidence.")

Cited source IDs: {'E6', 'E7', 'E5', 'E1', 'E8', 'E2'}
Missing source IDs: set()
All citation IDs refer to retrieved evidence.


In [88]:
citation_table = pd.DataFrame([
    {
        "source_id": item["source_id"],
        "document": item["document"],
        "page": item["page"],
        "passage": item["text"]
    }
    for item in evidence
])

display(citation_table)

,source_id,document,page,passage
0,E1,microsoft_annual_report_2025.pdf,21,and Results of Operations” in our Annual Repor...
1,E2,microsoft_annual_report_2025.pdf,72,"71 \nRevenue, classified by significant produc..."
2,E3,microsoft_annual_report_2025.pdf,25,"ating Income $ 14,166 $ 11,959 ..."
3,E4,microsoft_annual_report_2025.pdf,22,generally higher in the fourth quarter of our ...
4,E5,microsoft_annual_report_2025.pdf,24,growth \n \nRevenue from Azure and other clo...
5,E6,microsoft_annual_report_2025.pdf,23,ity and Business Processes and Intelligent Clo...
6,E7,microsoft_annual_report_2025.pdf,72,"rvices 7,404 6,648 6,417 ..."
7,E8,microsoft_annual_report_2025.pdf,42,41 \nService and other revenue includes sales ...


In [89]:
citation_table.to_csv(
    OUTPUT_DIR / "retrieval_audit.csv",
    index=False
)

print("Retrieval evidence saved.")

Retrieval evidence saved.


In [90]:
print("Embedding shape:", embedding_matrix.shape)
print("Largest absolute value:", np.max(np.abs(embedding_matrix)))
print("All values finite:", np.isfinite(embedding_matrix).all())
print("Zero rows:", np.sum(np.all(embedding_matrix == 0, axis=1)))

Embedding shape: (2296, 768)
Largest absolute value: 0.2377745
All values finite: True
Zero rows: 0


In [91]:
def safe_normalize(vector):

    vector = np.asarray(vector, dtype=np.float64)

    if not np.isfinite(vector).all():
        raise ValueError("Embedding contains NaN or infinity.")

    max_value = np.max(np.abs(vector))

    if max_value == 0:
        raise ValueError("Embedding is a zero vector.")

    scaled = vector / max_value

    norm = np.sqrt(np.sum(scaled * scaled))

    if not np.isfinite(norm) or norm == 0:
        raise ValueError("Invalid embedding norm.")

    return scaled / norm

In [92]:
valid_indices = []

normalized_vectors = []

for index, vector in enumerate(embedding_matrix):

    try:
        normalized_vectors.append(safe_normalize(vector))
        valid_indices.append(index)

    except ValueError:
        print("Skipping invalid embedding:", index)

chunks_df = chunks_df.iloc[valid_indices].reset_index(drop=True)

normalized_embeddings = np.vstack(normalized_vectors)

print("Normalized shape:", normalized_embeddings.shape)
print("All normalized values finite:", np.isfinite(normalized_embeddings).all())
print("Maximum vector norm:", np.max(np.linalg.norm(normalized_embeddings, axis=1)))

Normalized shape: (2296, 768)
All normalized values finite: True
Maximum vector norm: 1.0000000000000002


In [93]:
def search_financial_documents(question, top_k=5):

    query_embedding = get_query_embedding(question)

    query_embedding = safe_normalize(query_embedding)

    similarities = np.sum(
        normalized_embeddings * query_embedding,
        axis=1
    )

    if not np.isfinite(similarities).all():
        raise ValueValueError("Invalid similarity scores.")

    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = chunks_df.iloc[top_indices].copy()

    results["similarity_score"] = similarities[top_indices]

    return results[
        [
            "document_name",
            "page_number",
            "chunk_index",
            "chunk_text",
            "similarity_score"
        ]
    ].reset_index(drop=True)

In [94]:
search_results = search_financial_documents(
    "What are Microsoft's major revenue streams?",
    top_k=8
)

display(search_results)

assert np.isfinite(
    search_results["similarity_score"]
).all()

print("Retrieval test passed.")

,document_name,page_number,chunk_index,chunk_text,similarity_score
0,microsoft_annual_report_2025.pdf,24,1,growth \n \nRevenue from Azure and other clo...,0.742819
1,microsoft_annual_report_2025.pdf,72,0,"71 \nRevenue, classified by significant produc...",0.734748
2,microsoft_annual_report_2025.pdf,21,1,and Results of Operations” in our Annual Repor...,0.732983
3,microsoft_annual_report_2025.pdf,21,3,mics 365 revenue growth of \n19%. \n• Server ...,0.722755
4,microsoft_annual_report_2025.pdf,25,2,"ating Income $ 14,166 $ 11,959 ...",0.722663
5,microsoft_annual_report_2025.pdf,15,1,"d in-game content), Xbox Game Pass and other s...",0.712259
6,microsoft_annual_report_2025.pdf,42,0,41 \nService and other revenue includes sales ...,0.711252
7,microsoft_annual_report_2025.pdf,21,2,devices. Our most significant expenses are rel...,0.710950


Retrieval test passed.


In [95]:
def build_verification_prompt(answer, evidence):

    context = format_evidence(evidence)

    prompt = f"""
You are a financial evidence verifier.

Compare the answer with the supplied evidence.

For every factual claim in the answer:
- Mark SUPPORTED if the evidence directly supports it.
- Mark UNSUPPORTED if the evidence does not establish it.
- Mark CONTRADICTED if the evidence conflicts with it.
- Identify the source IDs that support or contradict the claim.
- Do not use outside knowledge.
- Do not treat a citation ID as proof by itself.

Return a numbered list with:
Claim:
Status:
Evidence:
Reason:

ANSWER:
{answer}

EVIDENCE:
{context}
"""
    return prompt

In [96]:
def verify_statementiq_answer(answer, evidence):

    prompt = build_verification_prompt(
        answer,
        evidence
    )

    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            "model": LLM_MODEL,
            "prompt": prompt,
            "stream": False
        },
        timeout=600
    )

    response.raise_for_status()

    return response.json()["response"]

In [97]:
question = "What are Microsoft's major revenue streams?"

answer, retrieved_evidence, evidence = ask_statementiq_v2(
    question,
    top_k=8
)

print("ORIGINAL ANSWER")
print(answer)

ORIGINAL ANSWER
Microsoft's major revenue streams are:

**Cloud and Server Services**
*   Server products and cloud services [E2]
*   Microsoft Cloud revenue [E8]
*   Azure and other cloud services, which include cloud and AI consumption-based services, GitHub cloud services, Nuance Healthcare cloud services, virtual desktop offerings, and other cloud services [E1, E4, E8]
*   Dynamics products and cloud services [E2, E7, E8]

**Productivity and Software Licensing**
*   Microsoft 365 Commercial products and cloud services [E2, E5, E8]
*   Microsoft 365 Consumer products and cloud services [E2, E8]
*   Licensing and supporting an array of software products [E3]
*   Office 365 [E7]

**Devices and Hardware**
*   Windows and Devices [E2]
*   Windows OEM and Devices revenue, derived from the sales of Windows Pro and non-Pro licenses sold through the OEM channel, and sales of first-party Devices, such as Surface and PC accessories [E1, E2, E4, E8]
*   Designing and selling devices [E3]

**Ad

In [98]:
verification_result = verify_statementiq_answer(
    answer,
    evidence
)

print("CLAIM VERIFICATION")
print(verification_result)

CLAIM VERIFICATION
1. Claim: Server products and cloud services
Status: SUPPORTED
Evidence: E2
Reason: E2 explicitly lists "Server products and cloud services" as a revenue category.

2. Claim: Microsoft Cloud revenue
Status: SUPPORTED
Evidence: E8
Reason: E8 lists "Microsoft Cloud revenue increased 23% to $168.9 billion," confirming this segment.

3. Claim: Azure and other cloud services, which include cloud and AI consumption-based services, GitHub cloud services, Nuance Healthcare cloud services, virtual desktop offerings, and other cloud services
Status: SUPPORTED
Evidence: E1, E4, E8
Reason: E1 provides a comprehensive list of these services included in the revenue. E4 and E8 confirm that the revenue for "Server products and cloud services" is driven by "Azure and other cloud services."

4. Claim: Dynamics products and cloud services
Status: SUPPORTED
Evidence: E2, E7, E8
Reason: E2 lists "Dynamics products and cloud services" as a revenue offering. E7 mentions sales from "Dynamic

In [99]:
verification_path = (
    OUTPUT_DIR / "claim_verification_report.txt"
)

with open(verification_path, "w", encoding="utf-8") as file:

    file.write("QUESTION\n")
    file.write(question + "\n\n")

    file.write("ORIGINAL ANSWER\n")
    file.write(answer + "\n\n")

    file.write("CLAIM VERIFICATION\n")
    file.write(verification_result)

print("Verification report saved:", verification_path)

Verification report saved: /Users/naitikdonda/Desktop/StatementIQ/outputs/claim_verification_report.txt
